# Retrieval Stage

In [1]:
import sys
import os
%load_ext autoreload
%autoreload 2

sys.path.append(os.path.abspath("../src"))

## Spark + Data Initialisation

In [2]:
from utils.spark_session import get_spark
import configs.settings as cfg

spark = get_spark()
spark.sparkContext.setLogLevel("INFO")


# load csv as dataframe
ratings = spark.read.csv("../data/raw/rating.csv", schema=cfg.RATINGS_SCHEMA, header=True)
ratings.printSchema()
ratings.show(5)

root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- rating: float (nullable = true)
 |-- timestamp: timestamp (nullable = true)

+------+-------+------+-------------------+
|userId|movieId|rating|          timestamp|
+------+-------+------+-------------------+
|     1|      2|   3.5|2005-04-02 23:53:47|
|     1|     29|   3.5|2005-04-02 23:31:16|
|     1|     32|   3.5|2005-04-02 23:33:39|
|     1|     47|   3.5|2005-04-02 23:32:07|
|     1|     50|   3.5|2005-04-02 23:29:40|
+------+-------+------+-------------------+
only showing top 5 rows


### Data Split
To minimise data leakage, training data will be split chronologically so that future user preferences aren't erroneously used to train predictions.

Each user will be represented in both the training and test dataset. This allows evaluation on test data to compare to retrieval and ranking lists.

NB: The high min. number of ratings per user of 20 allows this and avoids cold-start issues.

In [3]:
# Split data + cache training data for performance
from data.split_data import chron_user_tt_split

train, test = chron_user_tt_split(ratings, cfg.USER_COL, cfg.TIMESTAMP_COL, 0.8)

In [4]:
# Filter test results to only show test items of rating 4+
test_filtered = test.filter(test.rating >= 4)

## Retrieval Stage
Narrowing down millions of items to 100 - 500 for ranking. Experiments will be logged with MLflow to compare hyperparameters and performance.

### MLflow Setup

In [5]:
# Setting up experiment
import mlflow
mlflow.set_tracking_uri('file:../mlruns')
mlflow.set_experiment("ALS_Retrieval")

<Experiment: artifact_location=('file:///Users/davidtambiah/Data Projects/Movie '
 'Recommender/recsys_prod/notebooks/../mlruns/577512861714272474'), creation_time=1772681715171, experiment_id='577512861714272474', last_update_time=1772681715171, lifecycle_stage='active', name='ALS_Retrieval', tags={}>

### ALS
Computing ALS to reduce model items from millions to hundreds for ranking stage.

In [6]:
# Setting up config file for ALS
config = {
    'alpha': 3,
    'maxIter': 10,
    'rank': 50,
    'implicitPrefs': True,
    'regParam': 0.2,
    'coldStartStrategy':'drop',
    'nonnegative': False,
    'seed': 2026,
    'userCol': cfg.USER_COL,
    'itemCol': cfg.ITEM_COL,
    'ratingCol': cfg.RATING_COL
}

# Number of recommendations per user & for evaluation cut-off
top_k = 100

# Generate run name for mlflow
run_name = (
    f"alpha{config['alpha']}"
    f"_rank{config['rank']}"
    f"_maxIter{config['maxIter']}"
    f"_regParam{config['regParam']}"
    f"_topK{top_k}"
)

In [8]:
from retrieval import als
from evaluation.ranking import prepare_eval_df, compute_ranking_metrics

# Running experiment through MLFlow
with mlflow.start_run(run_name=run_name):
    mlflow.log_param('alpha', config['alpha'])
    mlflow.log_param('rank', config['rank'])
    mlflow.log_param('maxIter', config['maxIter'])
    mlflow.log_param('regParam', config['regParam'])
    mlflow.log_param('top_k', top_k)
    
    # Generating and fitting ALS model from config and train df
    model = als.train(train, config)
    mlflow.spark.log_model(model, 
                           artifact_path='als_model',
                          input_example=train.limit(1).toPandas())
    
    # Generating k candidates based on model and training data
    # incl. post-processing to remove already seen items
    recs = als.generate_candidates(model, train, k=top_k)
    
    # Transforming recommendations into eval_df --> generating metrics
    eval_df = prepare_eval_df(recs, test_filtered, cfg.USER_COL, cfg.ITEM_COL)
    metrics = compute_ranking_metrics(eval_df, k=top_k)
    
    # Logging metrics (Precision@k, Recall@k, NDCG@k and MAP)
    for metric, score in metrics.items():
        mlflow.log_metric(metric, score)


/opt/anaconda3/envs/recsys/lib/python3.9/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/03/19 16:10:32 INFO mlflow.spark: File '/var/folders/wv/d8774jps5zzdnz1w9h0k6mj00000gn/T/tmp7umfm6ph/model/sparkml' is already on DFS, copy is not necessary.
/opt/anaconda3/envs/recsys/lib/python3.9/site-packages/pyspark/sql/con

## Final Model + Parameters
Highest on all metrics was alpha=3, maxIter=10, rank=50 and regParam=0.2. NDCG score was 0.165; MAP was 0.055.

In [72]:
# Saving final recs data as parquets
recs.write.mode('overwrite').parquet('../data/retrieval/als_candidates.parquet')

In [4]:
# Saving train and test data as parquet
train.write.mode('overwrite').parquet('../data/retrieval/train.parquet')
test_filtered.write.mode('overwrite').parquet('../data/retrieval/test_filtered.parquet')